# Day 3 · NumPy 进阶(2/3)

目标:索引与切片、布尔掩码、聚合、矩阵乘法。

今日节奏:40min 学习 + 15min 动手 + 5min 自检(最后有清单)。

> 打开方式:JupyterLab(http://127.0.0.1:8888/lab)文件树里进 `ai-learning/练习/` 双击本文件,逐格 Shift+Enter。

In [1]:
import sys
import numpy as np

print("Python", sys.version.split()[0], "| numpy", np.__version__)
print("环境 OK!开始今天的练习 →")

Python 3.14.6 | numpy 2.5.2
环境 OK!开始今天的练习 →


## 任务 1:索引与切片

一维数组的切片和 Python 列表一样:`arr[开始:结束:步长]`,但 numpy 的切片是**视图**(改切片会改原数组,这点和列表不同!)。

先猜再跑:
- `v[2]` → 下标 2 的元素
- `v[1:4]` → 下标 1..3(不含 4)
- `v[::-1]` → 反转
- `v[[0, 5, 8]]` → 花式索引:按下标列表取元素,返回**新数组**

In [2]:
v = np.arange(10)
print("v:", v)
print("v[2]:", v[2])
print("v[1:4]:", v[1:4])
print("v[::-1]:", v[::-1])
print("v[[0, 5, 8]]:", v[[0, 5, 8]])

v: [0 1 2 3 4 5 6 7 8 9]
v[2]: 2
v[1:4]: [1 2 3]
v[::-1]: [9 8 7 6 5 4 3 2 1 0]
v[[0, 5, 8]]: [0 5 8]


## 任务 2:二维索引(行、列、子块)

二维数组 `M[行, 列]`,逗号两边分别是行索引和列索引:

- `M[1]` → 第 1 行(所有列)
- `M[:, 2]` → 第 2 列(所有行)——昨天见过
- `M[1:3, 1:3]` → 中间 2×2 子块(行 1..2、列 1..2)
- `M[-1]` → 最后一行(View 提醒:切片结果要改原数组,先学会用 `.copy()` 避开坑)

In [5]:
M = np.arange(12).reshape(3, 4)
print("M:\n", M)
print("M[1]:", M[1])
print("M[:, 2]:", M[:, 2])
print("M[1:3, 1:3]:\n", M[1:3, 1:3])
print("M[-1]:", M[-1])
print("M[2, 3]:", M[2, 3])   # 单个元素:第2行第3列

M:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
M[1]: [4 5 6 7]
M[:, 2]: [ 2  6 10]
M[1:3, 1:3]:
 [[ 5  6]
 [ 9 10]]
M[-1]: [ 8  9 10 11]
M[2, 3]: 11


## 任务 3:布尔掩码(最实用的一招)

用一个**条件表达式**生成 True/False 数组,然后直接当索引用——这能过滤出符合条件的元素,数据处理天天用:

- `v > 5` → 布尔数组
- `v[v > 5]` → 只留下大于 5 的元素
- `M[M % 2 == 0]` → 挑出所有偶数(注意结果是一维的)

In [6]:
v = np.array([3, 7, 1, 9, 4, 6])
mask = v > 5
print("v:", v)
print("mask:", mask)
print("v[mask]:", v[mask])

M = np.arange(12).reshape(3, 4)
even = M[M % 2 == 0]
print("偶数:", even)

v: [3 7 1 9 4 6]
mask: [False  True False  True False  True]
v[mask]: [7 9 6]
偶数: [ 0  2  4  6  8 10]


## 任务 4:聚合与 axis(今天主菜)

聚合 = 把一堆数变成一个数:`sum` / `mean` / `max` / `min`。

重点理解 **axis**(昨天预热过):
- `M.sum()` → 全部元素之和
- `M.sum(axis=0)` → **沿行方向移动**,每一列出一个数 → 得到列和(结果长度=列数)
- `M.sum(axis=1)` → **沿列方向移动**,每一行出一个数 → 得到行和(结果长度=行数)

记法:`axis=k` 表示"沿第 k 个轴方向折叠",折叠后该轴消失。

In [7]:
M = np.arange(12).reshape(3, 4)
print("M:\n", M)
print("sum 全部:", M.sum())
print("sum axis=0(每列和):", M.sum(axis=0))
print("sum axis=1(每行和):", M.sum(axis=1))
print("mean axis=0(每列均值):", M.mean(axis=0))
print("max(全局):", M.max(), "| argmax(位置):", M.argmax())

M:
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
sum 全部: 66
sum axis=0(每列和): [12 15 18 21]
sum axis=1(每行和): [ 6 22 38]
mean axis=0(每列均值): [4. 5. 6. 7.]
max(全局): 11 | argmax(位置): 11


## 任务 5:矩阵乘法(深度学习的地基)

两种写法等价:
- `A @ B`(推荐,Python 3.5+)
- `np.matmul(A, B)` 或 `A.dot(B)`

规则:结果形状 = (A的行, B的列);要求 A 的列数 == B 的行数。

⚠️ 别和**逐元素乘** `A * B` 搞混:后者要求形状完全一致。

**思考题**:`M @ M.T` 和 `M * M.T` 分别是什么形状?先心算,再运行。

In [8]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])
print("A @ B(矩阵乘):\n", A @ B)
print("A * B(逐元素乘):\n", A * B)

M = np.arange(12).reshape(3, 4)
print("M @ M.T 形状:", (M @ M.T).shape)   # (3,4)@(4,3) → (3,3)
try:
    print("M * M.T:", M * M.T)
except ValueError as e:
    print("M * M.T 报错(形状不一致不能逐元素乘):", e)

A @ B(矩阵乘):
 [[19 22]
 [43 50]]
A * B(逐元素乘):
 [[ 5 12]
 [21 32]]
M @ M.T 形状: (3, 3)
M * M.T 报错(形状不一致不能逐元素乘): operands could not be broadcast together with shapes (3,4) (4,3) 


## 任务 6:小挑战 🔥

有一组学生成绩(5 人 × 3 门课),完成:
1. 打印每个人的总分(axis=1)
2. 打印每门课的平均分(axis=0)
3. 找出总分最高的人(索引)
4. 用布尔掩码找出所有及格(≥60)的分数

In [13]:
scores = np.array([[85, 92, 78],
                   [58, 66, 72],
                   [90, 88, 95],
                   [45, 55, 60],
                   [70, 63, 81]])
print("成绩单:\n", scores)
print("每人总分:", scores.sum(axis=1))
print("每科平均:", scores.mean(axis=0))
scores_sum = scores.sum(axis=1)
idx = scores_sum.argmax()
print("总分最高是第", idx + 1, "位(索引", idx, ")")
print("及格分数:", scores[scores >= 60])

成绩单:
 [[85 92 78]
 [58 66 72]
 [90 88 95]
 [45 55 60]
 [70 63 81]]
每人总分: [255 196 273 160 214]
每科平均: [69.6 72.8 77.2]
总分最高是第 3 位(索引 2 )
及格分数: [85 92 78 66 72 90 88 95 60 70 63 81]


## 自检清单(6 问,答不上就回看今天的格子)

1. `v[1:4]` 和 `v[[1,2,3]]` 的区别? → 前者是**视图**(改它影响原数组),后者是**新数组**
2. 视图 vs 副本? → 切片/reshape 是视图;花式索引、布尔掩码、`.copy()` 是新数组
3. `M[:, 1]` 取的是什么? → 第 1 列(所有行)
4. `v[v > 5]` 干了什么? → 布尔掩码过滤,返回大于 5 的元素
5. axis=0 / axis=1 的聚合方向? → axis=0 折叠行方向(每列一个结果);axis=1 折叠列方向(每行一个结果)
6. `A @ B` 与 `A * B`? → 矩阵乘法(形状 (A行,B列)) vs 逐元素乘(形状必须一致)

## 📝 收盘动作

```powershell
cd D:\01_Study\ai-learning
git add -A; git commit -m "day3: numpy advanced"; git push
```

然后跟助手说"生成日志"。